In [1]:
import pandas as pd
import numpy as np

import mosmapapi

In [2]:
# !pip install openpyxl

In [4]:
df = pd.read_excel("../data/moscow_transformed.xlsx").set_index(['ID на сайте', 'Источник'], drop=False)
df

,,ID на сайте,Источник,Название,Цена,Дата,Тип автора,Метро/Район,Адрес,lat,lng,URL,Ссылки на картинки,"Расстояние до метро, км",Этаж,Этажность здания,Вид объекта,Общая площадь
ID на сайте,Источник,,,,,,,,,,,,,,,,,
320731003,cian.ru,320731003,cian.ru,"Офис в Москва Шипиловская ул., 58к1 (25 м²)",38800,2025-08-23 02:23:27,Агентство,Шипиловская,"Шипиловская ул., 58к1",55.621211,37.745600,https://www.cian.ru/rent/commercial/320731003,['https://images.cdn-cian.ru/images/2592583348...,0.083,4.0,6.0,Офисное помещение,25.0
321876001,cian.ru,321876001,cian.ru,"Офис в Москва Ленинградский просп., 47С2 (25 м²)",31250,2025-09-25 20:21:36,Агентство,Аэропорт,"Ленинградский просп., 47С2",55.799036,37.532601,https://www.cian.ru/rent/commercial/321876001,['https://images.cdn-cian.ru/images/2631122267...,0.250,-2.0,7.0,Офисное помещение,25.0
322186790,cian.ru,322186790,cian.ru,"Офис в Москва Волгоградский просп., 2 (20 м²)",29200,2025-09-25 10:22:06,Агентство,Пролетарская,"Волгоградский просп., 2",55.731176,37.669279,https://www.cian.ru/rent/commercial/322186790,['https://images.cdn-cian.ru/images/2641248288...,0.250,6.0,16.0,Офисное помещение,20.0
321392086,cian.ru,321392086,cian.ru,Помещение свободного назначения в Москва ул. А...,32000,2025-09-02 04:23:10,Частное лицо,Бунинская Аллея,"ул. Адмирала Руднева, 20",55.540409,37.516153,https://www.cian.ru/rent/commercial/321392086,['https://images.cdn-cian.ru/images/nezhiloe-p...,0.250,3.0,7.0,Торговое / Свободного назначения,32.0
324943206,cian.ru,324943206,cian.ru,"Офис в Москва просп. Мира, 95С1 (24 м²)",44000,2025-12-11 16:28:22,Агентство,Алексеевская,"просп. Мира, 95С1",55.808002,37.635853,https://www.cian.ru/rent/commercial/324943206,['https://images.cdn-cian.ru/images/ofis-moskv...,0.167,14.0,17.0,Офисное помещение,24.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3987628968735107072,realty.yandex.ru,3987628968735107072,realty.yandex.ru,Офис (1200 м²),6000000,2025-09-05 16:00:52,Агентство,Шелепиха,"1-й Магистральный тупик, 5А",55.766132,37.531956,https://realty.ya.ru/offer/3987628968735107205/,NaN,11.917,4.0,8.0,Офисное помещение,1200.0
322424654,cian.ru,322424654,cian.ru,"Офис в Москва Сколковское ш., вл43 (1149 м²)",6894000,2025-10-03 08:21:53,Агентство,Кунцевская,"Сколковское ш., вл43",55.699654,37.397701,https://www.cian.ru/rent/commercial/322424654,['https://images.cdn-cian.ru/images/2650029058...,7.333,1.0,6.0,Офисное помещение,1149.0
7523548143730329600,realty.yandex.ru,7523548143730329600,realty.yandex.ru,Офис (967 м²),6769000,2025-10-22 07:59:36,Агентство,Немчиновка,"Сколковское шоссе, вл43",55.699654,37.397700,https://realty.ya.ru/offer/7523548143730329381/,['https://avatars.mds.yandex.net/get-realty-of...,18.333,3.0,6.0,Офисное помещение,967.0


In [5]:
new_df = pd.read_csv("../data/moscow_super_transformed.csv")
if (len(new_df) < 10):
    new_df = pd.DataFrame()
else:
    new_df = new_df.set_index(['ID на сайте', 'Источник'])
existing_indices = [set(new_df.index), set(new_df.index)]

def get_batch_data(l: int, r: int, radius: int, id: int): # [l, r)
    global new_df
    ls = []
    for i in range(l, r):
        __ = (df.iloc[i]['ID на сайте'], df.iloc[i]['Источник'])
        if __ in existing_indices[id]:
            continue
        lat = df.iloc[i]['lat']
        lng = df.iloc[i]['lng']
        _ =pd.DataFrame(mosmapapi.get_data_radius(lat, lng,radius), index=[0])
        _['ID на сайте'] = df.iloc[i]['ID на сайте']
        _['Источник'] = df.iloc[i]['Источник']
        existing_indices[id].add(__)
        ls.append(_)
    if len(ls) == 0:
        return None
    return pd.concat(ls).set_index(['ID на сайте', 'Источник'])

def get_all_data(batch_size: int, l: int, r: int):
    global new_df
    for i in range(l, r, batch_size):
        dt1 = get_batch_data(i, min(i + batch_size, r), 300, 0)
        dt2 = get_batch_data(i, min(i + batch_size, r), 600, 1)
        if dt1 is None or dt2 is None:
            continue
        _ = pd.concat([dt1, dt2.drop(['district_price', 'district_name'], axis=1)], axis=1)
        new_df = pd.concat([new_df, _], axis=0)
        new_df.to_csv("../data/moscow_super_transformed.csv")

In [6]:
existing_indices

[{(325113244, 'cian.ru'),
  (323203178, 'cian.ru'),
  (323573244, 'cian.ru'),
  (323281388, 'cian.ru'),
  (8660284132703721472, 'realty.yandex.ru'),
  (3086750354037853184, 'realty.yandex.ru'),
  (323168366, 'cian.ru'),
  (321149121, 'cian.ru'),
  (6819069171518081024, 'realty.yandex.ru'),
  (325265827, 'cian.ru'),
  (321389462, 'cian.ru'),
  (323351575, 'cian.ru'),
  (320915497, 'cian.ru'),
  (321960386, 'cian.ru'),
  (4121682813316516864, 'realty.yandex.ru'),
  (3086799692142487040, 'realty.yandex.ru'),
  (320448588, 'cian.ru'),
  (324775102, 'cian.ru'),
  (321280430, 'cian.ru'),
  (323162150, 'cian.ru'),
  (321878137, 'cian.ru'),
  (319367485, 'cian.ru'),
  (321391222, 'cian.ru'),
  (7080657422337197056, 'realty.yandex.ru'),
  (324552914, 'cian.ru'),
  (325158265, 'cian.ru'),
  (8660284132706432000, 'realty.yandex.ru'),
  (325182999, 'cian.ru'),
  (324639802, 'cian.ru'),
  (7517866235140775936, 'realty.yandex.ru'),
  (324645707, 'cian.ru'),
  (322640033, 'cian.ru'),
  (322876529, 'c

In [7]:
new_df

,,district_name,n_buildings_300m,n_living_buildings_300m,n_flats_300m,min_bc_distance_300m,mean_bc_distance_300m,traffic1_300m,traffic2_300m,traffic3_300m,traffic4_300m,...,Ветаптеки и ветклиники_600,Магазины цветов_600,"Прачечные, химчистки_600",Детские игровые залы_600,Религия_600,"Пиццерии, суши, столовые_600","Пекарни, кофейни_600","Кафе, бары, рестораны_600",Социальные_600,Банки_600
ID на сайте,Источник,,,,,,,,,,,,,,,,,,,,,
320731003,cian.ru,Зябликово,38.0,14.0,3610.0,0.0,123.500000,8176.0,6091.0,1557.0,1713.0,...,8,8,7,0,0,3,7,11,8,1
321876001,cian.ru,Хорошевский,54.0,15.0,1287.0,0.0,82.666667,9392.0,38428.0,1513.0,11424.0,...,3,12,9,0,0,10,34,30,5,2
322186790,cian.ru,Таганский,44.0,22.0,3234.0,9.0,178.200000,8304.0,19462.0,2032.0,6787.0,...,10,17,15,0,1,11,29,29,10,2
321392086,cian.ru,Бутово Южное,23.0,8.0,1831.0,10.0,10.000000,4194.0,4593.0,2101.0,3820.0,...,1,6,3,0,1,4,7,9,5,0
324943206,cian.ru,Останкинский,46.0,30.0,3647.0,4.0,117.000000,7146.0,24972.0,4008.0,17350.0,...,6,8,15,0,1,16,27,29,10,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
324035526,cian.ru,Гольяново,20.0,14.0,4010.0,157.0,157.000000,5896.0,4311.0,12891.0,588.0,...,2,7,5,0,0,8,13,18,0,0
320759234,cian.ru,Зюзино,22.0,10.0,861.0,48.0,133.000000,1018.0,40175.0,5384.0,28894.0,...,2,5,2,0,1,6,1,6,2,1
322153703,cian.ru,Останкинский,15.0,2.0,97.0,27.0,27.000000,1508.0,5622.0,2812.0,4538.0,...,2,4,3,6,0,5,15,44,4,0


In [8]:
df.iloc[795][['lat', 'lng']]

lat    55.814593
lng    37.693785
Name: (323409428, cian.ru), dtype: object

In [9]:
mosmapapi.api_call_analytic(55.814593, 37.7456, 300)

{'latitude': 55.814593,
 'longitude': 37.7456,
 'district_name': 'Метрогородок',
 'price': {'district_price': '283200',
  'district_price_room1': '293800',
  'district_price_room2': '277400',
  'district_price_room3': '274800',
  'district_price_room4': '243600'},
 'orgs': {'2': {'group_name': 'Медицина', 'count': 0},
  '3': {'group_name': 'Стоматологии', 'count': 0},
  '4': {'group_name': 'Аптеки, оптики', 'count': 0},
  '5': {'group_name': 'Салоны красоты', 'count': 0},
  '6': {'group_name': 'Бани, сауны', 'count': 0},
  '7': {'group_name': 'Фитнес-центры, тренажерные', 'count': 0},
  '8': {'group_name': 'Супермаркеты', 'count': 0},
  '9': {'group_name': 'Гипермаркеты', 'count': 0},
  '10': {'group_name': 'Продуктовые магазины', 'count': 0},
  '11': {'group_name': 'Алкомаркеты, магазины пива', 'count': 0},
  '13': {'group_name': 'Школы, лицеи, гимназии', 'count': 0},
  '14': {'group_name': 'Детские сады', 'count': 0},
  '15': {'group_name': 'Колледжи, институты', 'count': 0},
  '16':

In [10]:
mosmapapi.get_data_radius(55.621211, 37.7456, 300)

{'district_name': 'Зябликово',
 'n_buildings_300m': 38,
 'n_living_buildings_300m': 14,
 'n_flats_300m': 3610,
 'min_bc_distance_300m': np.int64(0),
 'mean_bc_distance_300m': np.float64(123.5),
 'traffic1_300m': 8176,
 'traffic2_300m': 6091,
 'traffic3_300m': 1557,
 'traffic4_300m': 1713,
 'Медицина_300': 3,
 'Стоматологии_300': 3,
 'Аптеки, оптики_300': 4,
 'Салоны красоты_300': 13,
 'Бани, сауны_300': 0,
 'Фитнес-центры, тренажерные_300': 1,
 'Супермаркеты_300': 0,
 'Гипермаркеты_300': 0,
 'Продуктовые магазины_300': 10,
 'Алкомаркеты, магазины пива_300': 3,
 'Школы, лицеи, гимназии_300': 2,
 'Детские сады_300': 3,
 'Колледжи, институты_300': 0,
 'Студенческие общежития_300': 0,
 'Торговые центры, моллы_300': 0,
 'Бизнес-центры_300': 1,
 'Пункты выдачи товаров_300': 8,
 'Ветаптеки и ветклиники_300': 0,
 'Магазины цветов_300': 1,
 'Прачечные, химчистки_300': 2,
 'Детские игровые залы_300': 0,
 'Религия_300': 0,
 'Пиццерии, суши, столовые_300': 0,
 'Пекарни, кофейни_300': 1,
 'Кафе, ба

In [12]:
get_all_data(10, 0, 4500)

C:\Users\misha\AppData\Local\Temp\ipykernel_6384\716075108.py:24: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(ls).set_index(['ID на сайте', 'Источник'])
C:\Users\misha\AppData\Local\Temp\ipykernel_6384\716075108.py:24: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(ls).set_index(['ID на сайте', 'Источник'])
C:\Users\misha\AppData\Local\Temp\ipykernel_6384\716075108.py:24: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is dep

KeyboardInterrupt: 